# Notebook 02 — Feature Engineering

**Project:** BudgetWise — Expense Tracker & Budget Planner with ML  
**Purpose:** Transform raw transaction data into features suitable for:
1. **Expense Forecasting** — predict next month's spend per category
2. **Category Classification** — predict category from description text
3. **Anomaly Detection** — compute statistical baselines per category

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('../data/sample/demo_transactions.csv', parse_dates=['date'])
df.head()

## 1. Time Features (for Forecasting)

In [ ]:
df['year']         = df['date'].dt.year
df['month']        = df['date'].dt.month
df['day']          = df['date'].dt.day
df['day_of_week']  = df['date'].dt.dayofweek   # 0=Monday
df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
df['quarter']      = df['date'].dt.quarter

# Monthly total per category — used as label for forecasting
expenses = df[df['type'] == 'expense'].copy()
monthly_cat = (
    expenses
    .groupby(['year', 'month', 'category'])['amount']
    .sum()
    .reset_index()
    .rename(columns={'amount': 'monthly_total'})
)

# Lag features: previous month spend per category
monthly_cat = monthly_cat.sort_values(['category', 'year', 'month'])
monthly_cat['lag_1'] = monthly_cat.groupby('category')['monthly_total'].shift(1)
monthly_cat['lag_2'] = monthly_cat.groupby('category')['monthly_total'].shift(2)
monthly_cat['rolling_mean_3'] = (
    monthly_cat.groupby('category')['monthly_total']
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

monthly_cat = monthly_cat.dropna()
print(f'Forecasting feature set shape: {monthly_cat.shape}')
monthly_cat.head(10)

## 2. Text Features (for Category Classification)

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_description'] = df['description'].apply(clean_text)

# TF-IDF vectorisation
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_text = tfidf.fit_transform(df['clean_description'])
print(f'TF-IDF feature matrix shape: {X_text.shape}')

# Category labels
le = LabelEncoder()
y_category = le.fit_transform(df['category'])
print('Classes:', le.classes_)

## 3. Anomaly Detection — Statistical Baselines

In [ ]:
# Compute mean and std per expense category
stats = (
    expenses
    .groupby('category')['amount']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'mean_amount', 'std': 'std_amount'})
)

# Z-score for each transaction
expenses = expenses.merge(stats, on='category')
expenses['z_score'] = (expenses['amount'] - expenses['mean_amount']) / expenses['std_amount'].replace(0, np.nan)

anomalies = expenses[expenses['z_score'].abs() >= 2.5]
print(f'Anomalies detected: {len(anomalies)}')
anomalies[['date', 'description', 'category', 'amount', 'z_score']]

## 4. Save Processed Datasets

Save the engineered datasets to `../data/processed/` for use in training.

In [ ]:
monthly_cat.to_csv('../data/processed/forecasting_features.csv', index=False)
df[['clean_description', 'category']].to_csv('../data/processed/classification_data.csv', index=False)
stats.to_csv('../data/processed/anomaly_baselines.csv')

print('Processed datasets saved to ../data/processed/')